In [ ]:
!pip install transformers
!pip install datasets
!pip install torch
!pip install pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 18.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [ ]:
import pandas as pd
import torch
from transformers import BartTokenizer, BartForConditionalGeneration
from torch.utils.data import Dataset, DataLoader


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [ ]:
import pandas as pd

# Load the sarcastic and non-sarcastic datasets
sarcastic_df = pd.read_csv('Final_sarcastic_data.csv')
non_sarcastic_df = pd.read_csv('Final_non_sarcastic_data.csv')

# Display the first few rows of each dataset
print("Sarcastic Data:")
print(sarcastic_df.head())

print("\nNon-Sarcastic Data:")
print(non_sarcastic_df.head())


Sarcastic Data:
   id                                          sarcastic
0   1  Oh great, another meeting! My life is complete...
1   2  Wow, this traffic jam is *just* what I needed ...
2   3  Sure, giving me extra work at 5 PM is such a *...
3   4  Fantastic, my laptop decided to update in the ...
4   5  Your cooking is so good it belongs on a disast...

Non-Sarcastic Data:
   id                                    non_sarcastic
0   1                I dislike having another meeting.
1   2  This traffic jam is frustrating and unexpected.
2   3    Giving me extra work at 5 PM is inconvenient.
3   4  My laptop's update interrupted my presentation.
4   5                I don't like how the food tastes.


In [ ]:
# Merge the datasets on the 'id' column
merged_df = pd.merge(sarcastic_df, non_sarcastic_df, on='id')

# Rename columns for clarity
merged_df.columns = ['id', 'sarcastic', 'non_sarcastic']

# Display the merged dataset
print("\nMerged Dataset:")
print(merged_df.head())



Merged Dataset:
   id                                          sarcastic  \
0   1  Oh great, another meeting! My life is complete...   
1   2  Wow, this traffic jam is *just* what I needed ...   
2   3  Sure, giving me extra work at 5 PM is such a *...   
3   4  Fantastic, my laptop decided to update in the ...   
4   5  Your cooking is so good it belongs on a disast...   

                                     non_sarcastic  
0                I dislike having another meeting.  
1  This traffic jam is frustrating and unexpected.  
2    Giving me extra work at 5 PM is inconvenient.  
3  My laptop's update interrupted my presentation.  
4                I don't like how the food tastes.  


In [ ]:
# Check for null values
print("\nMissing Values:")
print(merged_df.isnull().sum())

# Drop rows with null values (if any)
merged_df = merged_df.dropna()

# Display the cleaned dataset
print("\nCleaned Dataset:")
print(merged_df.head())



Missing Values:
id               0
sarcastic        4
non_sarcastic    4
dtype: int64

Cleaned Dataset:
   id                                          sarcastic  \
0   1  Oh great, another meeting! My life is complete...   
1   2  Wow, this traffic jam is *just* what I needed ...   
2   3  Sure, giving me extra work at 5 PM is such a *...   
3   4  Fantastic, my laptop decided to update in the ...   
4   5  Your cooking is so good it belongs on a disast...   

                                     non_sarcastic  
0                I dislike having another meeting.  
1  This traffic jam is frustrating and unexpected.  
2    Giving me extra work at 5 PM is inconvenient.  
3  My laptop's update interrupted my presentation.  
4                I don't like how the food tastes.  


In [ ]:
merged_df.to_csv('cleaned_dataset.csv', index=False)
print("\nCleaned dataset saved as 'cleaned_dataset.csv'.")



Cleaned dataset saved as 'cleaned_dataset.csv'.


In [ ]:
# Ensure all text columns are strings
merged_df['sarcastic'] = merged_df['sarcastic'].astype(str)
merged_df['non_sarcastic'] = merged_df['non_sarcastic'].astype(str)

# Verify the data types
print("\nData Types:")
print(merged_df.dtypes)



Data Types:
id                int64
sarcastic        object
non_sarcastic    object
dtype: object


In [ ]:
from sklearn.model_selection import train_test_split

# Split the dataset (80% training, 20% validation)
train_df, val_df = train_test_split(merged_df, test_size=0.2, random_state=42)

print("\nTraining Set:")
print(train_df.head())

print("\nValidation Set:")
print(val_df.head())



Training Set:
      id                                          sarcastic  \
84    85   Oh good, my pen stopped working during the exam.   
10    11         Great, another email about the same issue.   
619  620  If you could kick the person in the pants resp...   
252  253  Sure, let�s schedule a call at the most inconv...   
896  897  Oh, of course, let’s just drop the project hal...   

                                         non_sarcastic  
84             My pen stopped working during the exam.  
10   Another email about the same issue is unnecess...  
619  We often have ourselves to blame for our troub...  
252    The call is scheduled for an inconvenient time.  
896  Following through with a project is necessary ...  

Validation Set:
      id                                          sarcastic  \
871  872  Oh, of course, let’s just spend all our time o...   
441  442  More disappointment, because life�s not hard e...   
344  345  How convenient, the app crashed and lost all m..

In [ ]:
from transformers import BartTokenizer

# Load the BART tokenizer
tokenizer = BartTokenizer.from_pretrained('facebook/bart-large')

# Tokenize the datasets
def tokenize_data(df):
    inputs = tokenizer(df['sarcastic'].tolist(), max_length=128, truncation=True, padding=True, return_tensors="pt")
    labels = tokenizer(df['non_sarcastic'].tolist(), max_length=128, truncation=True, padding=True, return_tensors="pt")
    return inputs, labels

# Tokenize training and validation datasets
train_inputs, train_labels = tokenize_data(train_df)
val_inputs, val_labels = tokenize_data(val_df)

print("\nTokenization Complete.")


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.63k [00:00<?, ?B/s]


Tokenization Complete.


In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset

# Create a custom Dataset class
class SarcasmDataset(Dataset):
    def __init__(self, inputs, labels):
        self.inputs = inputs
        self.labels = labels

    def __len__(self):
        return len(self.inputs['input_ids'])

    def __getitem__(self, idx):
        return {
            'input_ids': self.inputs['input_ids'][idx],
            'attention_mask': self.inputs['attention_mask'][idx],
            'labels': self.labels['input_ids'][idx]
        }

# Create DataLoader objects for training and validation
train_dataset = SarcasmDataset(train_inputs, train_labels)
val_dataset = SarcasmDataset(val_inputs, val_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

print("\nDataLoaders are ready.")



DataLoaders are ready.


In [ ]:
from transformers import BartForConditionalGeneration

# Load pre-trained BART model for conditional generation
model = BartForConditionalGeneration.from_pretrained('facebook/bart-large')

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"Model loaded and moved to {device}.")


pytorch_model.bin:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

Model loaded and moved to cuda.


In [ ]:
from transformers import AdamW

# Define the optimizer
optimizer = AdamW(model.parameters(), lr=5e-5)

# Set training parameters
num_epochs = 3  # You can increase this for better results


/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [ ]:
from tqdm import tqdm

# Training function
def train_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0
    for batch in tqdm(dataloader, desc="Training"):
        # Move data to the device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Zero out gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        # Backward pass
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(dataloader)

# Validation function
def validate_epoch(model, dataloader, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validation"):
            # Move data to the device
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Forward pass
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            total_loss += loss.item()
    return total_loss / len(dataloader)


In [ ]:
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")

    # Train and validate
    train_loss = train_epoch(model, train_loader, optimizer, device)
    val_loss = validate_epoch(model, val_loader, device)

    print(f"Training Loss: {train_loss:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")

# Save the trained model
model.save_pretrained("sarcasm_to_non_sarcasm_bart")
print("\nModel saved as 'sarcasm_to_non_sarcasm_bart'.")



Epoch 1/3


Validation: 100%|██████████| 12/12 [00:01<00:00,  9.40it/s]


Training Loss: 5.8421
Validation Loss: 2.9687

Epoch 2/3


Validation: 100%|██████████| 12/12 [00:01<00:00,  8.88it/s]


Training Loss: 1.9550
Validation Loss: 1.0944

Epoch 3/3


Validation: 100%|██████████| 12/12 [00:01<00:00,  8.64it/s]
/usr/local/lib/python3.10/dist-packages/transformers/modeling_utils.py:2817: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


Training Loss: 0.5967
Validation Loss: 0.9336

Model saved as 'sarcasm_to_non_sarcasm_bart'.


In [ ]:
def generate_non_sarcastic(model, tokenizer, sarcastic_sentences, device, max_length=128):
    model.eval()
    generated_sentences = []

    for sentence in sarcastic_sentences:
        # Tokenize input
        inputs = tokenizer(sentence, return_tensors="pt", truncation=True, padding=True).to(device)

        # Generate non-sarcastic output
        outputs = model.generate(inputs.input_ids, max_length=max_length, num_beams=5, early_stopping=True)

        # Decode output tokens
        generated_sentence = tokenizer.decode(outputs[0], skip_special_tokens=True)
        generated_sentences.append(generated_sentence)

    return generated_sentences


In [ ]:
# Extract sarcastic sentences from the validation set
sarcastic_sentences = val_df['sarcastic'].tolist()

# Generate non-sarcastic predictions
predicted_sentences = generate_non_sarcastic(model, tokenizer, sarcastic_sentences, device)

# Add predictions to the validation DataFrame
val_df['predicted_non_sarcastic'] = predicted_sentences

# Display a few examples
print(val_df[['sarcastic', 'non_sarcastic', 'predicted_non_sarcastic']].head())


                                             sarcastic  \
871  Oh, of course, let’s just spend all our time o...   
441  More disappointment, because life�s not hard e...   
344  How convenient, the app crashed and lost all m...   
741  I just love when I can waste my entire day on ...   
790  Oh, perfect, another ‘surprise’ task that we d...   

                                         non_sarcastic  \
871  We should focus on the most important tasks fi...   
441       Life is bringing more disappointment my way.   
344            The app crashed and I lost my progress.   
741  I spent a lot of time on something unimportant...   
790   A new task has appeared that we didn’t plan for.   

                               predicted_non_sarcastic  
871  We need to complete important tasks to complet...  
441           More disappointment is weighing me down.  
344              The app crashed and lost my progress.  
741   I wasted my entire day on something unimportant.  
790              

In [ ]:
from nltk.translate.bleu_score import sentence_bleu

# Compute BLEU score for each prediction
val_df['bleu_score'] = val_df.apply(
    lambda row: sentence_bleu([row['non_sarcastic'].split()], row['predicted_non_sarcastic'].split()), axis=1
)

# Display average BLEU score
average_bleu = val_df['bleu_score'].mean()
print(f"\nAverage BLEU Score: {average_bleu:.4f}")



Average BLEU Score: 0.1200


/usr/local/lib/python3.10/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.10/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.10/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_

In [ ]:
!pip install rouge-score
from rouge_score import rouge_scorer

# Initialize the ROUGE scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Compute ROUGE scores for the predictions
rouge_scores = []
for pred, target in zip(val_df['predicted_non_sarcastic'], val_df['non_sarcastic']):
    scores = scorer.score(pred, target)
    rouge_scores.append(scores)

# Average the scores across the dataset
avg_rouge1 = sum(score['rouge1'].fmeasure for score in rouge_scores) / len(rouge_scores)
avg_rouge2 = sum(score['rouge2'].fmeasure for score in rouge_scores) / len(rouge_scores)
avg_rougeL = sum(score['rougeL'].fmeasure for score in rouge_scores) / len(rouge_scores)

print("\nAverage ROUGE Scores:")
print(f"ROUGE-1: {avg_rouge1:.4f}")
print(f"ROUGE-2: {avg_rouge2:.4f}")
print(f"ROUGE-L: {avg_rougeL:.4f}")



Average ROUGE Scores:
ROUGE-1: 0.4938
ROUGE-2: 0.2974
ROUGE-L: 0.4700


In [ ]:
val_df.to_csv("validation_results.csv", index=False)
print("\nValidation results saved as 'validation_results.csv'.")

# Download the file (for Google Colab)
from google.colab import files
files.download("validation_results.csv")



Validation results saved as 'validation_results.csv'.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>